# Main Notebook - Cross-Modal Temporal Prediction with RoBERTa–GPT-2 and CLIP-Guided Diffusion
---

**Introduction:**

In this notebook you will find sequence predictor capable of story coninuation

In [ ]:
!pip install torchinfo
!pip install clip
!pip install evaluate
!pip install rouge_score
!pip install nltk
!pip install diffusers
!pip install lpips

In [ ]:
# @title Importing the necessary libraries

import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
from bs4 import BeautifulSoup
import re
import matplotlib.pyplot as plt
import numpy as np
import os
from nltk.translate.bleu_score import sentence_bleu
import json
import pandas as pd
from torchinfo import summary
from transformers import CLIPProcessor, CLIPModel, RobertaModel, RobertaTokenizer
import evaluate
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import tqdm
from datasets.fingerprint import random
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.transforms.functional as FT
import math
from transformers import BertTokenizer
import gc
import random
import re
from typing import Dict, Any, List, Optional, Tuple
import textwrap

# **The data preparation**


---



## 1.1 Loading and saving data

In [ ]:
from src.utils.helper import load_checkpoint_from_drive, save_checkpoint_to_drive
# This will prompt you to authorize Google Drive access
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


We need to define a couple of functions to make our life easier. Feel free to tweak those functions:

In [ ]:
from src.utils.helper import (
    # Functions
    parse_gdi_text,
    show_image,
    save_checkpoint_to_drive,
    load_checkpoint_from_drive,
    _parse_markdown_table,
    parse_cot_grounding,
    extract_cot_text_for_frame,

    # Hyperparameters
    emb_dim,
    latent_dim,
    num_layers,
    max_seq_len,
    batch_size,
    dropout,

    # Datasets
    SequencePredictionDataset,
    TextTaskDataset,
    AutoEncoderTaskDataset,
)
emb_dim = emb_dim
latent_dim = latent_dim
num_layers = num_layers
max_seq_len = max_seq_len
batch_size = batch_size
dropout = dropout
spatial_dim=256
unfreeze_layers=2

device = device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Now we load dataset from HuggingFace:

In [ ]:
# @title Loading the dataset
from datasets import load_dataset

train_dataset = load_dataset("daniel3303/StoryReasoning", split="train")
test_dataset = load_dataset("daniel3303/StoryReasoning", split="test")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3552 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/626 [00:00<?, ? examples/s]

In [ ]:
# @title CoT improvements toggles
# Turn these on/off to control the 4 optional improvements.
USE_FRAME_AWARE_GROUNDING = True      # Option 2: align ROI to matching frame text embedding (instead of always frame 0)
USE_CONTRASTIVE_ROI = True            # Option 1: InfoNCE-style contrastive grounding using batch negatives
USE_ENTITY_POOLING = True             # Option 3: entity-specific pooling/consistency across batch by entity_id
USE_COT_TEXT = True                   # Option 4: concatenate CoT text snippet to the frame descriptions

# Contrastive temperature (only used if USE_CONTRASTIVE_ROI=True)
CONTRASTIVE_TAU = 0.07

## 1.3 Creating and testing our dataset objects and loaders


---



In [ ]:
# @title For the Sequence prediction task

from transformers import RobertaTokenizer, GPT2Tokenizer

enc_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
dec_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
dec_tokenizer.pad_token = dec_tokenizer.eos_token        # GPT-2 has no pad token

sp_train_dataset = SequencePredictionDataset(train_dataset, enc_tokenizer, dec_tokenizer)
sp_test_dataset  = SequencePredictionDataset(test_dataset,  enc_tokenizer, dec_tokenizer)


train_size = int(0.8 * len(sp_train_dataset))
val_size = len(sp_train_dataset) - train_size
train_subset, val_subset = random_split(sp_train_dataset, [train_size, val_size])

# Instantiate the dataloaders
train_dataloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
# We will use the validation set to visualize the progress.
val_dataloader = DataLoader(val_subset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(sp_test_dataset, batch_size=batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# @title For the text task
"""
Sets up the data pipeline for the auxiliary text autoencoding task.
Creates the `TextTaskDataset` and its corresponding `DataLoader`.
"""
enc_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
dec_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
dec_tokenizer.pad_token = dec_tokenizer.eos_token          # GPT-2 has no pad token

text_dataset = TextTaskDataset(train_dataset, enc_tokenizer, dec_tokenizer, max_len=max_seq_len)
text_dataloader = DataLoader(text_dataset, batch_size=batch_size, shuffle=True)

for batch in text_dataloader:
    print(batch["enc_input_ids"].shape, batch["enc_attention_mask"].shape, batch['target_ids'].shape, batch['target_attention_mask'])
    break

torch.Size([16, 128]) torch.Size([16, 128])


In [ ]:
print(train_dataset[0].keys())

dict_keys(['story_id', 'images', 'frame_count', 'chain_of_thought', 'story'])


In [ ]:
# @title For the image autoencoder task
"""
Sets up the data pipeline for the auxiliary visual autoencoding task.
Creates the `AutoEncoderTaskDataset` and its `DataLoader`.
"""

autoencoder_dataset = AutoEncoderTaskDataset(train_dataset)
autoencoder_dataloader = DataLoader(autoencoder_dataset, batch_size=batch_size, shuffle=True)

# **Models**


---



In [ ]:
from src.models.text_autoencoder import RobertaEncoder, TransformerDecoder, Seq2Seq, SinusoidalPositionalEncoding
from src.models.visual_autoencoder  import (
    ReconstructionLoss,
    VisualAutoencoder,
   init_weights,
)
from src.models.multimodal_predictor import SequencePredictor

## 3.1 Initialization and setup

In [ ]:
# @title Initializing the NLP models

torch.cuda.empty_cache()
gc.collect()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dec_tokenizer.pad_token = dec_tokenizer.eos_token      # GPT-2 has no pad token

text_autoencoder = Seq2Seq(
    encoder_name="roberta-base",
    decoder_name="gpt2",
    unfreeze_encoder_layers=6,
    enc_tokenizer=enc_tokenizer,
    dec_tokenizer=dec_tokenizer,
).to(device)

trainable = sum(p.numel() for p in text_autoencoder.parameters() if p.requires_grad)
total     = sum(p.numel() for p in text_autoencoder.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Total params:     {total:,}")
text_autoencoder, _, _, _ = load_checkpoint_from_drive(text_autoencoder, None, filename='text_autoencoder.pth')

trainable_params = sum(p.numel() for p in text_autoencoder.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in text_autoencoder.parameters())

print(f"Trainable params: {trainable_params}")
print(f"Total params: {total_params}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 123668736
Total params: 219962880


In [ ]:
# @title Initializing visual models
"""
1. Instantiates the `VisualAutoencoder`.
2. Applies the custom weight initialization.
"""

visual_autoencoder = VisualAutoencoder(
    latent_dim=128,
    spatial_dim=spatial_dim,
    unfreeze_layers=unfreeze_layers,
).to(device)
visual_autoencoder.decoder.apply(init_weights)

total_params = sum(p.numel() for p in visual_autoencoder.parameters() if p.requires_grad)


visual_autoencoder, _, _, _ = load_checkpoint_from_drive(visual_autoencoder, None, filename='visual_autoencoder.pth')
print(f"Total trainable parameters in visual autoencoder: {total_params}")

config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Checkpoint loaded from: /content/drive/MyDrive/DL_Checkpoints/visual_autoencoder.pth (epoch 14)
Total trainable parameters in visual autoencoder: 46405699


In [ ]:
import inspect
from src.training.train_predictor import (
    train_sequence_predictor, evaluate_predictor,
)


In [ ]:
sequence_predictor = SequencePredictor(
    visual_autoencoder, text_autoencoder, hidden_dim=512
).to(device)

# sequence_predictor, _, _, _, _ = load_predictor_checkpoint(
#     sequence_predictor, optimizer, filename='sequence_predictor.pth')
total_params = sum(p.numel() for p in sequence_predictor.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
print(f"Total parameters: {sum(p.numel() for p in sequence_predictor.parameters()):,}")

## 3.2 Training loops

In [ ]:
from src.models.visual_autoencoder import ReconstructionLoss

reconstruction_loss = ReconstructionLoss(
    pixel_weight=0.55,
    perceptual_weight=0.25,
    latent_weight=0.20,
    z_weight=1.0,
    spatial_weight=0.5,
).to(device)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, sequence_predictor.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)
history = train_sequence_predictor(
    predictor=sequence_predictor,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    optimizer=optimizer,
    reconstruction_loss=reconstruction_loss,
    enc_tokenizer=enc_tokenizer,
    dec_tokenizer=dec_tokenizer,
    device=device,
    epochs=20,
    checkpoint_dir="/content/drive/MyDrive/DL_Checkpoints",
    resume=True,
    text_weight=1.0,
    image_weight=1.0,
    attn_balance_weight=0.05,
    copy_margin_weight=0.10,
    last_frame_cap=0.55,
    use_context_dropout=True,
)

In [ ]:
from tests.test_predictor import (test_predictor_shapes,
                                  evaluate_predictor_test,
                                  plot_temporal_attention,
                                  frame4_copy_diagnostic,
                                  visualize_predictions,)


test_predictor_shapes(
    predictor=sequence_predictor,
    dataloader=test_dataloader,
    device=device,
)

metrics = evaluate_predictor_test(
    predictor=sequence_predictor,
    dataloader=test_dataloader,
    dec_tokenizer=dec_tokenizer,
    device=device,
    reconstruction_loss=reconstruction_loss,
)

plot_temporal_attention(
    predictor=sequence_predictor,
    dataloader=test_dataloader,
    device=device,
)

frame4_copy_diagnostic(
    predictor=sequence_predictor,
    dataloader=test_dataloader,
    device=device,
)

visualize_predictions(
    predictor=sequence_predictor,
    dataloader=test_dataloader,
    enc_tokenizer=enc_tokenizer,
    dec_tokenizer=dec_tokenizer,
    device=device,
)